In [1]:
from pathlib import Path # IMPORT PATH INTO JUPYTER
import pandas as pd

file = Path.home() / "OneDrive/Documents/GitHub/work_to_show/Kaggle/Agriculture/crop-yield.csv" # <- PATH IN JUPYTER
df = pd.read_csv(file) # READ FILE NAME
df.head() # sanity check

,N,P,K,Soil_pH,Soil_Moisture,Soil_Type,Organic_Carbon,Temperature,Humidity,Rainfall,Sunlight_Hours,Wind_Speed,Region,Altitude,Season,Crop_Type,Irrigation_Type,Fertilizer_Used,Pesticide_Used,Crop_Yield_ton_per_hectare
0,132,62,22,6.35,59.78,Clay,0.43,22.97,53.89,1305.68,7.73,15.96,Central,36,Rabi,Maize,Canal,223.48,23.36,11.42
1,122,71,66,5.98,25.54,Sandy,0.65,17.00,76.90,1942.05,9.25,12.60,North,1561,Rabi,Potato,Canal,161.54,4.42,23.19
2,44,35,104,8.07,25.87,Sandy,0.79,25.52,44.78,2216.20,8.50,15.63,North,1870,Rabi,Rice,Rainfed,184.62,6.29,7.94
3,136,96,113,4.83,42.97,Silt,0.45,18.59,31.89,607.18,8.75,5.49,East,765,Kharif,Sugarcane,Rainfed,274.02,2.72,72.53
4,101,34,42,5.84,48.01,Silt,0.69,22.74,46.27,483.47,8.00,7.44,Central,1143,Zaid,Wheat,Rainfed,72.69,15.37,6.72


In [3]:
df.dtypes

N                               int64
P                               int64
K                               int64
Soil_pH                       float64
Soil_Moisture                 float64
Soil_Type                      object
Organic_Carbon                float64
Temperature                   float64
Humidity                      float64
Rainfall                      float64
Sunlight_Hours                float64
Wind_Speed                    float64
Region                         object
Altitude                        int64
Season                         object
Crop_Type                      object
Irrigation_Type                object
Fertilizer_Used               float64
Pesticide_Used                float64
Crop_Yield_ton_per_hectare    float64
r                             float64
dtype: object

# Questions:
1. Which environmental conditions most strongly predict crop yield?
2. Does irrigation type and fertilizer use significantly improve yield across different soil types?
3. Which crop type performs best across different regions and seasons?

# question 1

In [4]:
# make factors
# Season                         object
# Crop_Type                      object
# Irrigation_Type                object

df["Season"] = df["Season"].astype("category") # if working in pandas
# df["Season"] = df["Season"].asfactor() if not working in pandas

df["Crop_Type"] = df["Crop_Type"].astype("category") # if working in pandas
# df["Crop_Type"] = df["Crop_Type"].asfactor() if not working in pandas

df["Irrigation_Type"] = df["Irrigation_Type"].astype("category") # if working in pandas
# df["Irrigation_Type"] = df["Irrigation_Type"].asfactor() if not working in pandas

df.dtypes

N                                int64
P                                int64
K                                int64
Soil_pH                        float64
Soil_Moisture                  float64
Soil_Type                       object
Organic_Carbon                 float64
Temperature                    float64
Humidity                       float64
Rainfall                       float64
Sunlight_Hours                 float64
Wind_Speed                     float64
Region                          object
Altitude                         int64
Season                        category
Crop_Type                     category
Irrigation_Type               category
Fertilizer_Used                float64
Pesticide_Used                 float64
Crop_Yield_ton_per_hectare     float64
r                              float64
dtype: object

In [5]:
print(df.columns.tolist())

['N', 'P', 'K', 'Soil_pH', 'Soil_Moisture', 'Soil_Type', 'Organic_Carbon', 'Temperature', 'Humidity', 'Rainfall', 'Sunlight_Hours', 'Wind_Speed', 'Region', 'Altitude', 'Season', 'Crop_Type', 'Irrigation_Type', 'Fertilizer_Used', 'Pesticide_Used', 'Crop_Yield_ton_per_hectare', 'r']


In [6]:
# Encode all categorical string columns to numeric codes
cat_cols = ['Soil_Type', 'Region', 'Season', 'Crop_Type', 'Irrigation_Type']

for col in cat_cols:
    df[col] = df[col].astype("category").cat.codes   # converts strings to numeric codes

In [7]:
# Which environmental conditions most strongly predict crop yield?
# RandomForestRegressor
VALUES = ['N', 'P', 'K', 'Soil_pH', 'Soil_Moisture', 'Soil_Type', 'Organic_Carbon', 'Temperature', 'Humidity', 'Rainfall', 'Sunlight_Hours', 'Wind_Speed']           # list of input feature column names
TARGET_COL = "Crop_Yield_ton_per_hectare"                                      # column we are trying to predict

# Divide data into train and test sets
import numpy as np
np.random.seed(42)                                           # set seed so random results are reproducible
df['r'] = np.random.uniform(size=len(df))                   # add column of random numbers (0 to 1) to each row
train = df[df["r"] <= .6]                                    # ~60% of rows go to training set
test = df[df["r"] > .6]                                      # ~40% of rows go to test set

# Random Forest in Python
from sklearn.metrics import mean_squared_error, r2_score     # regression evaluation metrics
from sklearn.ensemble import RandomForestRegressor           # regressor instead of classifier

X = train[VALUES]                                            # training features (inputs)
y = train[TARGET_COL]                                        # training target (what we predict)
Xtest = test[VALUES]                                         # test features (inputs)
ytest = test[TARGET_COL]                                     # test target (what we predict)

rf = RandomForestRegressor(n_estimators=500,                 # build 500 decision trees
                            random_state=17)                 # seed for reproducibility
rf.fit(X, y)                                                 # train the model on training data

# # Predict
# predictions = rf.predict(Xtest)                              # use trained model to predict on test set
# print("MSE: ", mean_squared_error(ytest, predictions))       # avg squared error (lower is better)
# print("R²:  ", r2_score(ytest, predictions))                 # 1.0 = perfect fit, 0 = bad fit

# # Make a prediction using the random forest
# newX = [[1,1,1,0,0]]                                         # one new data point with 5 feature values
# print(rf.predict(newX))                                      # predict the numeric value for this new row

# Additional Benefit of Random Forest Models: Feature Selection
importances = rf.feature_importances_                        # get importance score for each feature
sorted_indices = np.argsort(importances)[::-1]               # sort indices from most to least important
feat_labels = X.columns[0:]                                  # grab feature column names

for f in range(X.shape[1]):                                  # loop through each feature
    print("%2d) %-*s %f" % (f + 1, 30,                      # print rank number
                             feat_labels[sorted_indices[f]], # print feature name
                             importances[sorted_indices[f]]))# print importance score

# Feature Importance Graphed
# from matplotlib import pyplot as plt
# plt.title('Feature Importance')                              # set chart title
# plt.bar(range(X.shape[1]),                                   # create bar for each feature
#         importances[sorted_indices], align='center')         # bar height = importance score
# plt.xticks(range(X.shape[1]),                                # set x-axis tick positions
#            X.columns[sorted_indices], rotation=90)           # label ticks with feature names, rotated
# plt.tight_layout()                                           # adjust layout so labels don't get cut off
# plt.show()                                                   # display the chart

 1) Rainfall                       0.097679
 2) Soil_Moisture                  0.095371
 3) Temperature                    0.094776
 4) Humidity                       0.093144
 5) Sunlight_Hours                 0.093105
 6) Wind_Speed                     0.092152
 7) Soil_pH                        0.087940
 8) N                              0.083109
 9) Organic_Carbon                 0.082365
10) K                              0.078969
11) P                              0.077930
12) Soil_Type                      0.023460


In [9]:
# multiple regression
#divide into training and test data sets
import numpy as np
np.random.seed(42)
df['r'] = np.random.uniform(size=len(df))
#re.head()
train=df[df["r"] <= .6]
#train.head()
test=df[df["r"] > .6]
#test.head()
train=train.drop("r",axis=1)
test=test.drop("r",axis=1)
#----------------------------------------------------------------------------
#%%
#variable selection
from sklearn.feature_selection import SelectFromModel
from sklearn.linear_model import LassoCV
from sklearn import preprocessing
X = train[VALUES]#Feature Matrix
y = train[TARGET_COL]#Target Variable
X = preprocessing.scale(X)
X=pd.DataFrame(X, columns =VALUES)
estimator = LassoCV(cv=5)
sfm = SelectFromModel(estimator, prefit=False, max_features=None,threshold="mean")
sfm.fit(X, y)
# =========================================
# See all coefficients before thresholding
estimator.fit(X, y)
coef_df = pd.DataFrame({
    'feature': X.columns,
    'coef': np.abs(estimator.coef_)
}).sort_values('coef', ascending=False)
print(coef_df)
print("Best alpha chosen:", estimator.alpha_)
# =========================================
feature_idx = sfm.get_support()
selected_features = X.columns[feature_idx]
print(selected_features)
#----------------------------------------------------------------------------
#%%
#fit the model to the training data
import statsmodels.api as sm
X=train[selected_features]
X=sm.add_constant(X)
reg = sm.OLS(y,X).fit()
print(reg.summary())
#check to see that Prob(F-statistic)<0.05, to see if the model is significant
#----------------------------------------------------------------------------
#%%
#evaluate the model on the test data
tX=test[selected_features]
tX=sm.add_constant(tX)
py=reg.predict(tX)
ty=test[TARGET_COL]
ae=abs(ty-py)
mae=np.mean(ae)
print("The mean absolute error is",mae)
N=len(test)
rmse=np.sqrt((np.sum((ty-py)**2))/N)
print("The root mean square error is",rmse)
#----------------------------------------------------------------------------
#%%
#make a prediction
#the first 1 is for the intercept
# newX= [[1,1200,2,1,3,120000]]
# reg.predict(newX)

           feature  coef
0                N   0.0
1                P   0.0
2                K   0.0
3          Soil_pH   0.0
4    Soil_Moisture   0.0
5        Soil_Type   0.0
6   Organic_Carbon   0.0
7      Temperature   0.0
8         Humidity   0.0
9         Rainfall   0.0
10  Sunlight_Hours   0.0
11      Wind_Speed   0.0
Best alpha chosen: 0.383844236692997
Index(['N', 'P', 'K', 'Soil_pH', 'Soil_Moisture', 'Soil_Type',
       'Organic_Carbon', 'Temperature', 'Humidity', 'Rainfall',
       'Sunlight_Hours', 'Wind_Speed'],
      dtype='object')
                                OLS Regression Results                                
Dep. Variable:     Crop_Yield_ton_per_hectare   R-squared:                       0.001
Model:                                    OLS   Adj. R-squared:                 -0.001
Method:                         Least Squares   F-statistic:                    0.5337
Date:                        Fri, 27 Mar 2026   Prob (F-statistic):              0.894
Time:         